In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- run_pipeline: local/interactive orchestrator
# =============================================================================
# Summary:      Chains Steps 1-4 (load/window -> train -> evaluate -> aggregate
#               metrics) for local/interactive development. For HPC/Slurm
#               submission, run each step notebook individually via
#               slurm/*.sbatch instead -- see this notebook's own markdown
#               cell below for why.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.0
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# run_pipeline — local/interactive orchestrator

Chains Steps 1 -> 2 -> 3 -> 4 in one call, each step's parameters set from
this notebook's own parameters cell below, using
`fumd_training_workflow.notebook_runner.run_step` (overwrite-the-
parameters-cell-then-execute -- the same trick `papermill` is built around,
implemented directly on `nbformat`/`nbclient` instead: see
`notebook_runner.py`'s module docstring for why this project doesn't take a
`papermill` dependency).

**This notebook is for local/interactive use.** This project's actual
execution environment submits **one notebook at a time** to Slurm
(`singularity exec ... jupyter execute <notebook>.ipynb`, see
`slurm/train_model.sbatch` / `slurm/evaluate_model.sbatch`) -- there is no
GPU-reachable way to run *this* notebook itself as one Slurm job and have
it dispatch each step to the GPU in turn. For an HPC run: run Step 1
(no GPU needed) locally or on a login node, submit Step 2 and Step 3 to
Slurm individually (edit their own parameters cells, or copy each notebook
per run the way this project's `sbatch` examples do), then run Step 4
locally once you have metrics to compare.

Out of the box this runs Steps 1-4 against the bundled tiny example
dataset with a small `EPOCHS` -- replace `DATASET_PATH` with a real
preprocessing-workflow `dataset_labeled_w<W>.csv` and raise `EPOCHS` for an
actual training run.

Steps 5 and 6 (BS-location estimation, migration-event maps) are not
chained here -- both need bring-your-own real-world data (see their own
notebooks' parameters cells) and neither needs a GPU.


In [ ]:
import os
import sys

sys.path.insert(0, "src")
from fumd_training_workflow import notebook_runner as nr

In [ ]:
# ---- Parameters ----
DATASET_PATH = "example-data/dataset_labeled_example.csv"
OUTPUT_DIR = "pipeline_run"
NOTEBOOKS_DIR = "notebooks"

SEQUENCE_LENGTH = 6
FUTURE_STEPS = [1, 2, 3, 4, 5, 6, 7]
VAL_FRACTION = 0.2
RANDOM_SEED = 42

EPOCHS = 5              # kept small so the bundled example trains in a couple of minutes; raise for a real run
BATCH_SIZE = 64
EARLY_STOPPING_PATIENCE = 3

# Optional: other dataset_labeled_w<W>.csv paths for Step 3's cross-run generalization check
CROSS_RUN_DATASET_PATHS = []

## Step 1 — load & window dataset

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_1_load_and_window_dataset.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_1_load_and_window_dataset.ipynb"),
    parameters={
        "DATASET_PATH": DATASET_PATH,
        "FEATURE_COLUMNS": None,
        "SEQUENCE_LENGTH": SEQUENCE_LENGTH,
        "FUTURE_STEPS": FUTURE_STEPS,
        "VAL_FRACTION": VAL_FRACTION,
        "RANDOM_SEED": RANDOM_SEED,
        "OUTPUT_DIR": OUTPUT_DIR,
    },
    cwd=os.getcwd(),
)
print("Step 1 done")

## Step 2 — train model

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_2_train_model.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_2_train_model.ipynb"),
    parameters={
        "INPUT_DIR": OUTPUT_DIR,
        "OUTPUT_DIR": OUTPUT_DIR,
        "EPOCHS": EPOCHS,
        "BATCH_SIZE": BATCH_SIZE,
        "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
        "RANDOM_SEED": RANDOM_SEED,
    },
    cwd=os.getcwd(),
)
print("Step 2 done")

## Step 3 — evaluate model

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_3_evaluate_model.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_3_evaluate_model.ipynb"),
    parameters={
        "RUN_DIR": OUTPUT_DIR,
        "METRICS_OUTPUT_DIR": OUTPUT_DIR,
        "CROSS_RUN_DATASET_PATHS": CROSS_RUN_DATASET_PATHS,
    },
    cwd=os.getcwd(),
)
print("Step 3 done")

## Step 4 — aggregate training metrics

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_4_aggregate_training_metrics.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_4_aggregate_training_metrics.ipynb"),
    parameters={
        "RUN_DIRS": [OUTPUT_DIR],
        "VARIANT_LABELS": None,
        "COMPARISON_OUTPUT_DIR": os.path.join(OUTPUT_DIR, "metrics_comparison"),
    },
    cwd=os.getcwd(),
)
print("Step 4 done")
print(f"pipeline complete -- see {OUTPUT_DIR}/ for the trained model, metrics, and comparison plots")